# Read Data_set_Max Workbook
Load selected sheets from `Data_set_Max.xlsx` and organize them into nested dictionaries.

## Import packages

In [ ]:
import Pkg

for pkg in ["XLSX", "DataFrames"]
    if Base.find_package(pkg) === nothing
        Pkg.add(pkg)
    end
end

using XLSX
using DataFrames
using CairoMakie

include("modules/gsw_rho.jl")

## Define functions

In [ ]:
xlsx_path = "Data_set_Max.xlsx"

sheet_sediment = "Sediment cores"
sheet_bottom_water = "Bottom water"
sheet_ctd = "CTD"

function read_sheet_df(path::AbstractString, sheet::AbstractString)
    return DataFrame(XLSX.readtable(path, sheet; infer_eltypes=true))
end

function normalize_colname(x)
    s = lowercase(String(x))
    s = replace(s, '_' => "", '-' => "", ' ' => "")
    return s
end

function find_col(df::DataFrame, candidates::Vector{String}; label::String)
    cols = names(df)
    norm_to_col = Dict(normalize_colname(c) => c for c in cols)

    for cand in candidates
        key = normalize_colname(cand)
        if haskey(norm_to_col, key)
            return norm_to_col[key]
        end
    end

    error("Could not find a $(label) column. Available columns: $(cols)")
end

function split_by_station(df::DataFrame; station_col=nothing)
    st_col = isnothing(station_col) ? find_col(df, ["station", "site", "stationid"], label="station") : station_col

    out = Dict{String,DataFrame}()
    for g_st in groupby(df, st_col)
        st = string(first(g_st[!, st_col]))
        out[st] = DataFrame(g_st)
    end
    return out
end

function split_sediment_cores(df::DataFrame; station_col=nothing, core_col=nothing)
    st_col = isnothing(station_col) ? find_col(df, ["station", "site", "stationid"], label="station") : station_col
    cr_col = isnothing(core_col) ? find_col(df, ["core", "coreid", "corenumber"], label="core") : core_col

    out = Dict{String,Dict{String,DataFrame}}()
    for g_st in groupby(df, st_col)
        st = string(first(g_st[!, st_col]))
        out[st] = Dict{String,DataFrame}()

        for g_cr in groupby(g_st, cr_col)
            cr = string(first(g_cr[!, cr_col]))
            out[st][cr] = DataFrame(g_cr)
        end
    end

    return out
end

## Read in sheets

In [ ]:
sediment_df = read_sheet_df(xlsx_path, sheet_sediment)
bottom_water_df = read_sheet_df(xlsx_path, sheet_bottom_water)
ctd_df = read_sheet_df(xlsx_path, sheet_ctd)

sed_cores = split_sediment_cores(sediment_df)
bottom_water = split_by_station(bottom_water_df)
ctd = split_by_station(ctd_df)

data_set_max = Dict(
    "sed_cores" => sed_cores,
    "bottom_water" => bottom_water,
    "ctd" => ctd,
)

## Show stations and cores (sediment)

In [ ]:
println("Sediment core stations: ", collect(keys(sed_cores)))
for st in sort(collect(keys(sed_cores)))
    println("  ", st, " cores: ", collect(keys(sed_cores[st])))
end

println("Bottom water stations: ", collect(keys(bottom_water)))
println("CTD stations: ", collect(keys(ctd)))

## Convert units to those used in RADI

Sediment cores - NO3, NH4, Ca, P, S, Fe, Mn, POC, PIC, H2S, FeS?, FeS2?

In [ ]:
# Calculate bottom water densities for each station
CS2_2_S = ctd["CS2-2"][end, 4]
CS2_2_T = ctd["CS2-2"][end, 3]
CS2_2_P = ctd["CS2-2"][end, 2]

HF2_S = ctd["HF2"][end, 4]
HF2_T = ctd["HF2"][end, 3]
HF2_P = ctd["HF2"][end, 2]

CS2_bottom_density = gsw_rho(CS2_2_S, CS2_2_T, CS2_2_P)
HF2_bottom_density = gsw_rho(HF2_S, HF2_T, HF2_P)

# Set molar masses (g/mol)
M_NO3 = 62.0049
M_NH4 = 18.0385
M_Ca = 40.078
M_PO4 = 94.9714
M_SO4 = 96.06
M_Fe = 55.845
M_Mn = 54.938
M_H2S = 34.0809
M_FeS = 87.91
M_FeS2 = 119.96

# Site-specific density lookup (kg/m^3)
site_density = Dict(
    "CS2-2" => CS2_bottom_density,
    "HF2" => HF2_bottom_density,
)

density_for_station(st::AbstractString) = get(site_density, st) do
    error("No bottom density defined for station: $(st)")
end

# Unit conversion helpers (numeric input only)
mgL_to_molkg_num(mgL::Real, molar_mass::Real, density::Real) = mgL / (molar_mass * density)
umolL_to_molkg_num(umolL::Real, density::Real) = umolL * 1e-3 / density
mmolL_to_molkg_num(mmolL::Real, density::Real) = mmolL / density
umolkg_to_molkg_num(umolkg::Real) = umolkg * 1e-6
umol_g_to_mol_kg_num(umol_g::Real) = umol_g * 1e-3

# Parse mixed cell values robustly
function to_float_or_missing(x)
    if ismissing(x)
        return missing
    elseif x isa Number
        return Float64(x)
    else
        p = tryparse(Float64, strip(String(x)))
        return p === nothing ? missing : p
    end
end

function convert_with_density(vals, rho, f::Function)
    out = Vector{Union{Missing, Float64}}(undef, length(vals))
    @inbounds for i in eachindex(vals)
        v = to_float_or_missing(vals[i])
        out[i] = ismissing(v) ? missing : f(v, rho[i])
    end
    return out
end

function convert_without_density(vals, f::Function)
    out = Vector{Union{Missing, Float64}}(undef, length(vals))
    @inbounds for i in eachindex(vals)
        v = to_float_or_missing(vals[i])
        out[i] = ismissing(v) ? missing : f(v)
    end
    return out
end

function add_converted_columns!(df::DataFrame; station_col_name::String="Station", include_extended::Bool=false)
    stations = string.(df[!, station_col_name])
    rho = [density_for_station(st) for st in stations]

    # Shared dissolved species conversions
    if "DIC (µmol/L)" in names(df)
        df[!, "DIC (mol/kg)"] = convert_with_density(df[!, "DIC (µmol/L)"], rho, umolL_to_molkg_num)
    end
    if "TA (µmol/kg)" in names(df)
        df[!, "TA (mol/kg)"] = convert_without_density(df[!, "TA (µmol/kg)"], umolkg_to_molkg_num)
    end
    if "NO3 (mg/L)" in names(df)
        df[!, "NO3 (mol/kg)"] = convert_with_density(df[!, "NO3 (mg/L)"], rho, (v, r) -> mgL_to_molkg_num(v, M_NO3, r))
    end
    if "NH4 (mg/L)" in names(df)
        df[!, "NH4 (mol/kg)"] = convert_with_density(df[!, "NH4 (mg/L)"], rho, (v, r) -> mgL_to_molkg_num(v, M_NH4, r))
    end

    # Sediment-only columns
    if include_extended
        if "Ca (mmol/L)" in names(df)
            df[!, "Ca (mol/kg)"] = convert_with_density(df[!, "Ca (mmol/L)"], rho, mmolL_to_molkg_num)
        end
        if "P (µmol/L)" in names(df)
            df[!, "P (mol/kg)"] = convert_with_density(df[!, "P (µmol/L)"], rho, umolL_to_molkg_num)
        end
        if "S (mmol/L)" in names(df)
            df[!, "S (mol/kg)"] = convert_with_density(df[!, "S (mmol/L)"], rho, mmolL_to_molkg_num)
        end
        if "Fe (µmol/L)" in names(df)
            df[!, "Fe (mol/kg)"] = convert_with_density(df[!, "Fe (µmol/L)"], rho, umolL_to_molkg_num)
        end
        if "Mn (µmol/L)" in names(df)
            df[!, "Mn (mol/kg)"] = convert_with_density(df[!, "Mn (µmol/L)"], rho, umolL_to_molkg_num)
        end
        if "H2S (µmol/L)" in names(df)
            df[!, "H2S (mol/kg)"] = convert_with_density(df[!, "H2S (µmol/L)"], rho, umolL_to_molkg_num)
        end
        if "FeS (µmol/g)" in names(df)
            df[!, "FeS (mol/kg)"] = convert_without_density(df[!, "FeS (µmol/g)"], umol_g_to_mol_kg_num)
        end
        if "FeS2 (µmol/g)" in names(df)
            df[!, "FeS2 (mol/kg)"] = convert_without_density(df[!, "FeS2 (µmol/g)"], umol_g_to_mol_kg_num)
        end
    end

    return df
end

# Apply conversions to original sheets
add_converted_columns!(sediment_df; include_extended=true)
add_converted_columns!(bottom_water_df; include_extended=false)

if "O2 (µmol/kg)" in names(ctd_df)
    ctd_df[!, "O2 (mol/kg)"] = convert_without_density(ctd_df[!, "O2 (µmol/kg)"], umolkg_to_molkg_num)
end

# Re-split from updated sheet DataFrames so nested dictionaries include the new columns
sed_cores = split_sediment_cores(sediment_df)
bottom_water = split_by_station(bottom_water_df)
ctd = split_by_station(ctd_df)

data_set_max = Dict(
    "sed_cores" => sed_cores,
    "bottom_water" => bottom_water,
    "ctd" => ctd,
)

println("Added converted columns to sediment_df, bottom_water_df, and ctd_df.")

In [ ]:
bottom_water["CS2-2"]

## Plot sediment profiles from both cores at each site

In [ ]:
sites = ["CS2-2", "HF2"]

for j in 1:2

    depths = sed_cores[sites[j]]["1"][:,4]

    var_labs = ["DIC (µmol/kg)", "TA (µmol/kg)", "NO3 (mg/L)", "NH4 (mg/L)", "Ca (mmol/L)", "PO4 (µmol/L)", "SO4 (mmol/L)",
        "Fe (µmol/L)", "Mn (µmol/L)", "POC (wt%)", "PIC (wt%)", "H2S (µmol/kg)", "FeS (µmol/g)", "FeS2 (µmol/g)"]

    var_cols = [5:15; 17:19]

    f = Figure()

    axes = Matrix{Axis}(undef, 2, 7)

    for i in 1:14
        row = (i <= 7) ? 1 : 2
        col = (i <= 7) ? i : i - 7
        axes[row, col] = Axis(f[row, col], xlabel = var_labs[i], ylabel = "Depth (cm)", yreversed = true, xaxisposition = :top, width = 130, height = 270)
        scatterlines!(axes[row, col], sed_cores[sites[j]]["1"][:, var_cols[i]], depths, color = :tomato, linestyle = :dash, marker = :circle, label="Core 1")
        scatterlines!(axes[row, col], sed_cores[sites[j]]["2"][:, var_cols[i]], depths, color = :blue, linestyle = :dash, marker = :circle, label="Core 2")
        # axislegend(axes[row, col], position = :rt)
    end

    linkyaxes!(axes...)

    Label(f[0, :], "Site $(sites[j])", fontsize = 28, halign = :center)
    resize_to_layout!(f)
    display(f)
end

## Plot porosity

In [ ]:
sites = ["CS2-2", "HF2"]
f = Figure()
axs = Axis[]  # use a name other than 'axes'

for j in 1:2
    depths = sed_cores[sites[j]]["1"][:,4]

    ax = Axis(f[1, j], xlabel = "Porosity", ylabel = "Depth (cm)", yreversed = true, xaxisposition = :top, width = 130, height = 270)
    push!(axs, ax)
    scatterlines!(ax, sed_cores[sites[j]]["1"][:, 16], depths, color = :tomato, linestyle = :dash, marker = :circle, label="Core 1")
    scatterlines!(ax, sed_cores[sites[j]]["2"][:, 16], depths, color = :blue, linestyle = :dash, marker = :circle, label="Core 2")

    Label(f[0, j], "Site $(sites[j])", fontsize = 28, halign = :center)
end

linkxaxes!(axs...)
linkyaxes!(axs...)
resize_to_layout!(f)
display(f)

## Fit porosity profile: phi(z) = phi_inf + (phi_0 - phi_inf)*exp(-b*z)

### Recommended fitting approach

Fit by minimizing squared errors in the original porosity space (not after log transform), with physically meaningful constraints:

- Use only valid depth-porosity pairs.
- Convert porosity to fraction if values are in percent (e.g. 70 -> 0.70).
- Constrain parameters to realistic ranges:
  - `0 < phi_inf < phi_0 < 1`
  - `b > 0`
- For each candidate `phi_inf`, solve for `A = phi_0 - phi_inf` and `b` from the linearized relation
  - `log(phi - phi_inf) = log(A) - b*z`
- Compute predictions in the original model and choose the parameter set with minimum SSE.

This gives stable, transparent fits without requiring external optimization packages.

In [ ]:
using Statistics

# Convert porosity values to fraction if they appear to be in percent.
to_porosity_fraction(v::Real) = v > 1.5 ? v / 100 : v

# Robust parser for mixed numeric/text/missing cells
function parse_num(x)
    if ismissing(x)
        return nothing
    elseif x isa Number
        xf = Float64(x)
        return isfinite(xf) ? xf : nothing
    else
        p = tryparse(Float64, strip(String(x)))
        return p === nothing || !isfinite(p) ? nothing : p
    end
end

# Build cleaned (z_mid, porosity_fraction) vectors
function clean_porosity_pairs(depth_from, depth_to, phi_raw)
    z = Float64[]
    phi = Float64[]
    for (df, dt, pv) in zip(depth_from, depth_to, phi_raw)
        dff = parse_num(df)
        dtt = parse_num(dt)
        pvv = parse_num(pv)
        if isnothing(dff) || isnothing(dtt) || isnothing(pvv)
            continue
        end
        zmid = (dff + dtt) / 2
        pf = to_porosity_fraction(pvv)
        if isfinite(zmid) && isfinite(pf)
            push!(z, zmid)
            push!(phi, pf)
        end
    end
    return z, phi
end

# Build depth weights that emphasize shallow samples.
# alpha > 0 gives exponentially decreasing weight with depth.
function shallow_weights(z; alpha::Float64=3.0)
    zmin, zmax = minimum(z), maximum(z)
    zr = zmax > zmin ? (z .- zmin) ./ (zmax - zmin) : zeros(length(z))
    w = exp.(-alpha .* zr)
    return w ./ mean(w)
end

# Fit with phi_0 fixed to shallowest observed porosity value, phi_inf varying.
# Uses weighted least squares when use_weights=true.
function fit_porosity_profile_fixed_phi0(z_raw, phi_raw; phi_inf_grid_size::Int=500, use_weights::Bool=false, alpha::Float64=3.0)
    n = length(z_raw)
    if n < 3
        return nothing
    end

    # Sort by depth so first means shallowest sample.
    perm = sortperm(z_raw)
    z = z_raw[perm]
    phi = phi_raw[perm]

    # Shift depth so shallowest point is z=0; this enforces phi(0)=phi_0 naturally.
    z = z .- minimum(z)

    eps = 1e-9
    phi_0 = phi[1]

    # Candidate range for phi_inf: physically below phi_0 and positive.
    lower = max(minimum(phi) * 0.2, eps)
    upper = min(phi_0 - eps, maximum(phi) - eps)
    if upper <= lower + eps
        return nothing
    end

    phi_inf_grid = range(lower, upper; length=phi_inf_grid_size)
    w_all = use_weights ? shallow_weights(z; alpha=alpha) : ones(length(z))

    best = nothing
    best_obj = Inf

    for phi_inf in phi_inf_grid
        A = phi_0 - phi_inf
        if A <= eps
            continue
        end

        # Need 0 < (phi - phi_inf)/A <= 1 for log transform.
        u = (phi .- phi_inf) ./ A
        if any(v -> v <= eps || v > 1 + 1e-8 || !isfinite(v), u)
            continue
        end

        ly = log.(u)

        # Through-origin weighted fit: ly = -b*z
        # b = - argmin sum w*(ly + b*z)^2
        denom = sum(w_all .* z .* z)
        if denom <= eps
            continue
        end

        b = -sum(w_all .* z .* ly) / denom
        if !(b > 0)
            continue
        end

        phi_pred = phi_inf .+ (phi_0 - phi_inf) .* exp.(-b .* z)
        if any(v -> !isfinite(v), phi_pred)
            continue
        end

        resid = phi .- phi_pred
        sse = sum(resid.^2)
        wsse = sum(w_all .* resid.^2)
        obj = use_weights ? wsse : sse

        if obj < best_obj
            rmse = sqrt(sse / n)
            wrmse = sqrt(wsse / sum(w_all))
            sst = sum((phi .- mean(phi)).^2)
            r2 = sst > eps ? 1 - sse / sst : NaN
            best_obj = obj
            best = (
                phi_0 = phi_0,
                phi_inf = phi_inf,
                b = b,
                sse = sse,
                wsse = wsse,
                rmse = rmse,
                wrmse = wrmse,
                r2 = r2,
                n = n,
                z = z,
                phi = phi,
                phi_pred = phi_pred,
                w = w_all
            )
        end
    end

    return best
end

sites = sort(collect(keys(sed_cores)))

function run_fit(; use_weights::Bool=false, alpha::Float64=3.0)
    out = DataFrame(
        Station = String[],
        Core = String[],
        phi_0 = Float64[],
        phi_inf = Float64[],
        b = Float64[],
        RMSE = Float64[],
        wRMSE = Float64[],
        R2 = Float64[],
        N = Int[]
)

    for st in sites
        cores = sort(collect(keys(sed_cores[st])))
        for cr in cores
            core_df = sed_cores[st][cr]
            z_mid, phi_vals = clean_porosity_pairs(
                core_df[!, "Depth_from (cm)"],
                core_df[!, "Depth_to (cm)"],
                core_df[!, "water (%)"]
            )

            fit = fit_porosity_profile_fixed_phi0(
                z_mid,
                phi_vals;
                use_weights=use_weights,
                alpha=alpha
            )

            if isnothing(fit)
                @warn "Could not fit porosity profile" station=st core=cr weighted=use_weights
                continue
            end

            push!(out, (
                st,
                string(cr),
                fit.phi_0,
                fit.phi_inf,
                fit.b,
                fit.rmse,
                fit.wrmse,
                fit.r2,
                fit.n
            ))
        end
    end

    sort!(out, [:Station, :Core])
    return out
end

# Current fit (unweighted) and new surface-prioritized weighted fit
porosity_fit_current = run_fit(use_weights=false)
porosity_fit_weighted = run_fit(use_weights=true, alpha=3.0)

println("Current fit (unweighted):")
display(porosity_fit_current)
println("Surface-prioritized weighted fit (alpha=3.0):")
display(porosity_fit_weighted)

# Comparison table for available cores
comparison = innerjoin(
    select(porosity_fit_current, :Station, :Core, :phi_0, :phi_inf, :b, :RMSE, :wRMSE, :R2),
    select(porosity_fit_weighted, :Station, :Core, :phi_0, :phi_inf, :b, :RMSE, :wRMSE, :R2),
    on=[:Station, :Core],
    makeunique=true
 )

rename!(comparison, Dict(
    :phi_0 => :phi_0_current,
    :phi_inf => :phi_inf_current,
    :b => :b_current,
    :RMSE => :RMSE_current,
    :wRMSE => :wRMSE_current,
    :R2 => :R2_current,
    :phi_0_1 => :phi_0_weighted,
    :phi_inf_1 => :phi_inf_weighted,
    :b_1 => :b_weighted,
    :RMSE_1 => :RMSE_weighted,
    :wRMSE_1 => :wRMSE_weighted,
    :R2_1 => :R2_weighted
))

comparison[!, :delta_phi_inf] = comparison.phi_inf_weighted .- comparison.phi_inf_current
comparison[!, :delta_b] = comparison.b_weighted .- comparison.b_current
comparison[!, :delta_RMSE] = comparison.RMSE_weighted .- comparison.RMSE_current
comparison[!, :delta_wRMSE] = comparison.wRMSE_weighted .- comparison.wRMSE_current

println("Weighted vs current fit comparison:")
display(comparison)

# Plot data, current fit, and weighted fit together.
f_fit = Figure()
axs_fit = Axis[]

for (j, st) in enumerate(sites)
    ax = Axis(
        f_fit[1, j],
        xlabel = "Porosity (fraction)",
        ylabel = "Depth (cm)",
        yreversed = true,
        xaxisposition = :top,
        width = 250,
        height = 320
    )
    push!(axs_fit, ax)

    for cr in sort(collect(keys(sed_cores[st])))
        core_df = sed_cores[st][cr]
        z_mid, phi_data = clean_porosity_pairs(
            core_df[!, "Depth_from (cm)"],
            core_df[!, "Depth_to (cm)"],
            core_df[!, "water (%)"]
        )

        row_cur = findfirst((porosity_fit_current.Station .== st) .& (porosity_fit_current.Core .== string(cr)))
        row_wgt = findfirst((porosity_fit_weighted.Station .== st) .& (porosity_fit_weighted.Core .== string(cr)))
        if isnothing(row_cur) || isnothing(row_wgt) || isempty(z_mid)
            continue
        end

        pcur = porosity_fit_current[row_cur, :]
        pwgt = porosity_fit_weighted[row_wgt, :]

        z_plot = range(minimum(z_mid), maximum(z_mid); length=150)
        z_plot_shift = z_plot .- minimum(z_mid)
        phi_cur = pcur.phi_inf .+ (pcur.phi_0 - pcur.phi_inf) .* exp.(-pcur.b .* z_plot_shift)
        phi_wgt = pwgt.phi_inf .+ (pwgt.phi_0 - pwgt.phi_inf) .* exp.(-pwgt.b .* z_plot_shift)

        c = cr == "1" ? :tomato : :blue
        scatter!(ax, phi_data, z_mid, color = c, marker = :circle, label = "Core $(cr) data")
        lines!(ax, phi_cur, z_plot, color = c, linestyle = :dash, linewidth = 2, label = "Core $(cr) current")
        lines!(ax, phi_wgt, z_plot, color = c, linestyle = :dot, linewidth = 2, label = "Core $(cr) weighted")
    end

    axislegend(ax, position = :rb)
    Label(f_fit[0, j], "Site $(st)", fontsize = 22, halign = :center)
end

linkyaxes!(axs_fit...)
resize_to_layout!(f_fit)
display(f_fit)

In [ ]:
println("Current vs weighted porosity fit summary:")
show(stdout, "text/plain", comparison)
println()

# Setting up CTD and bottom water data

In [ ]:
ctd_params = ["Temperature (°C)", "Salinity (PSU)", "Oxygen (µmol/kg)"]
sites = ["CS2-2", "HF2"]

f = Figure()
axs = Axis[]  # use a name other than 'axes'

for j in 1:3
    ax = Axis(f[1, j], xlabel = ctd_params[j], ylabel = "Depth (cm)", yreversed = true, xaxisposition = :top, width = 130, height = 270)
    push!(axs, ax)

    for i in 1:2
        depths = ctd[sites[i]][:, 2]
        scatterlines!(ax, ctd[sites[i]][:, j+2], depths, color = (i == 1) ? :tomato : :blue, linestyle = :dash, marker = :circle, label="Site $(sites[i])")
    end
end

display(f)

## Plot T, S DO for each site

In [ ]:
ctd_params = ["Temperature (°C)", "Salinity (PSU)", "Oxygen (µmol/kg)"]
sites = ["CS2-2", "HF2"]
f = Figure()
axs = Axis[]

for j in 1:3
    ax = Axis(f[1, j], xlabel = ctd_params[j], ylabel = "Depth (cm)", yreversed = true, xaxisposition = :top, width = 130, height = 270)
    push!(axs, ax)

    for i in 1:2
        raw_x = ctd[sites[i]][:, j+2]
        raw_d = ctd[sites[i]][:, 2]
        keep = [v isa Number && !ismissing(v) && d isa Number && !ismissing(d)
                for (v, d) in zip(raw_x, raw_d)]
        x = Float64.(raw_x[keep])
        d = Float64.(raw_d[keep])
        scatterlines!(ax, x, d, color = (i == 1) ? :tomato : :blue, linestyle = :dash, marker = :circle, label="Site $(sites[i])")
    end

    axislegend(ax, position = :rb)  # use `ax`, outside inner loop
end

linkyaxes!(axs...)
resize_to_layout!(f)
display(f)

## Plot DIC, TA, NO3, NH4

In [ ]:
using Statistics

bw_params = ["DIC (µmol/kg)", "TA (µmol/kg)", "NO3 (mg/L)", "NH4 (mg/L)"]
sites = ["CS2-2", "HF2"]
f = Figure()
axs = Axis[]

for j in 1:2
    ax = Axis(f[1, j], xlabel = bw_params[j], ylabel = "Depth (cm)", yreversed = true, xaxisposition = :top, width = 130, height = 270)
    push!(axs, ax)

    for i in 1:2
        df = bottom_water[sites[i]]
        
        # Group by column 2 (depth) and compute mean of column j+3
        result = combine(groupby(df, 2), j+3 => mean => :mean_val)
        
        # Extract columns (may contain Missing)
        depths_raw = result[!, 1]
        values_raw = result[!, :mean_val]
        
        # Filter for valid numeric pairs (not missing, not NaN)
        keep = [!ismissing(v) && !ismissing(d) && !isnan(v) && !isnan(d) 
                for (v, d) in zip(values_raw, depths_raw)]
        
        # Convert only the kept values
        x = Float64.(values_raw[keep])
        d = Float64.(depths_raw[keep])
        
        scatterlines!(ax, x, d, color = (i == 1) ? :tomato : :blue, linestyle = :dash, marker = :circle, label="Site $(sites[i])")
    end

    axislegend(ax, position = :rb)  # use `ax`, outside inner loop
end

for j in 3:4
    ax = Axis(f[1, j], xlabel = bw_params[j], ylabel = "Depth (cm)", yreversed = true, xaxisposition = :top, width = 130, height = 270)
    push!(axs, ax)
    
    for i in 1:2
        depths = bottom_water[sites[i]][1:3:end, 2]
        bw_param_values = bottom_water[sites[i]][1:3:end, j+3]

        scatterlines!(ax, bw_param_values, depths, color = (i == 1) ? :tomato : :blue, linestyle = :dash, marker = :circle, label="Site $(sites[i])")
    end

    axislegend(ax, position = :rb)  # use `ax`, outside inner loop
end

linkyaxes!(axs...)
resize_to_layout!(f)
display(f)